# DirectDebit IQ — Payment Success Analytics & Failure Predictor

## 01 — Exploratory Data Analysis

This notebook explores synthetic direct debit payment data to understand success patterns, failure drivers, merchant risk, customer risk, mandate-age impact, and geographic/banking segment performance.

**Business goal:** help payment operations, risk, and merchant success teams identify where payment failures are concentrated and which interventions can improve collection success.

---

## 1. Setup & Data Loading

This section loads the raw CSV dataset, connects to SQLite, runs the monthly SQL analysis, and performs basic data-quality checks.

In [1]:
from pathlib import Path
import sqlite3
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Use a notebook-friendly Plotly renderer.
# Change to "browser" if your local Jupyter environment does not show charts inline.
pio.renderers.default = "notebook_connected"


def find_project_root() -> Path:
    """Find the DirectDebit IQ project root from either root or notebooks/ execution."""
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw" / "payments.csv").exists() and (candidate / "sql").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/raw/payments.csv and sql/.")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "payments.csv"
DB_PATH = PROJECT_ROOT / "data" / "payments.db"
SQL_DIR = PROJECT_ROOT / "sql"

print(f"Project root: {PROJECT_ROOT}")
print(f"CSV path:      {DATA_PATH}")
print(f"SQLite DB:     {DB_PATH}")

Project root: /mnt/data/directdebit-iq
CSV path:      /mnt/data/directdebit-iq/data/raw/payments.csv
SQLite DB:     /mnt/data/directdebit-iq/data/payments.db


In [2]:
# Load CSV data
df = pd.read_csv(DATA_PATH, parse_dates=["payment_date"])

# Add analysis-friendly flags
df["is_success"] = (df["payment_status"] == "success").astype(int)
df["is_failed"] = (df["payment_status"] == "failed").astype(int)
df["year_month"] = df["payment_date"].dt.to_period("M").astype(str)
df["week_of_month"] = ((df["payment_day_of_month"] - 1) // 7 + 1).map(lambda x: f"Week {x}")

# Connect to SQLite and run the monthly success-rate SQL file
conn = sqlite3.connect(DB_PATH)
monthly_sql = (SQL_DIR / "01_monthly_success_rates.sql").read_text(encoding="utf-8")
monthly_success = pd.read_sql_query(monthly_sql, conn)
monthly_success["year_month_date"] = pd.to_datetime(monthly_success["year_month"] + "-01")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (50000, 20)


,payment_id,merchant_id,customer_id,payment_amount,currency,payment_date,payment_day_of_month,day_of_week,mandate_age_days,previous_failure_count,bank_country,bank_type,estimated_balance_band,days_since_last_success,payment_type,payment_status,is_success,is_failed,year_month,week_of_month
0,b5b778e9-7ff1-4020-89fe-82ad18ecbfad,M148,C0990,94.48,USD,2026-03-03,3,Tuesday,245,0,GB,challenger,medium,1,recurring,success,1,0,2026-03,Week 1
1,2d54c631-e2f9-4572-9192-6154bf0af3a8,M177,C4950,15.97,GBP,2024-11-27,27,Wednesday,311,0,GB,high_street,high,61,recurring,success,1,0,2024-11,Week 4
2,07f72a05-f7ae-4565-9544-ecddfcfe99c4,M115,C0208,64.61,GBP,2025-02-15,15,Saturday,139,0,GB,high_street,high,23,recurring,success,1,0,2025-02,Week 3
3,82a6adcf-ca64-4f30-8def-d5765a7e5ef2,M083,C1482,42.82,GBP,2025-07-28,28,Monday,42,0,GB,credit_union,low,37,recurring,success,1,0,2025-07,Week 4
4,dcdbbade-1f0b-4f49-b0c2-c78deb1aa72f,M001,C2832,251.21,GBP,2025-07-12,12,Saturday,1319,1,GB,high_street,high,31,recurring,success,1,0,2025-07,Week 2


In [3]:
# Data types and monthly SQL preview
display(Markdown("### Data Types"))
display(df.dtypes.to_frame("dtype"))

display(Markdown("### Monthly Success Rates from SQLite"))
display(monthly_success.head(10))

### Data Types

,dtype
payment_id,object
merchant_id,object
customer_id,object
payment_amount,float64
currency,object
payment_date,datetime64[ns]
payment_day_of_month,int64
day_of_week,object
mandate_age_days,int64
previous_failure_count,int64


### Monthly Success Rates from SQLite

,year_month,total_payments,successful_payments,failed_payments,success_rate_pct,vs_previous_month_change,year_month_date
0,2024-06,1960,1652,308,84.29,NaN,2024-06-01
1,2024-07,2156,1834,322,85.06,0.77,2024-07-01
2,2024-08,2034,1734,300,85.25,0.19,2024-08-01
3,2024-09,2011,1685,326,83.79,-1.46,2024-09-01
4,2024-10,2087,1798,289,86.15,2.36,2024-10-01
5,2024-11,2081,1802,279,86.59,0.44,2024-11-01
6,2024-12,2144,1834,310,85.54,-1.05,2024-12-01
7,2025-01,2129,1813,316,85.16,-0.38,2025-01-01
8,2025-02,1917,1644,273,85.76,0.60,2025-02-01
9,2025-03,2149,1848,301,85.99,0.23,2025-03-01


---

## 2. Payment Success Overview

This section gives a high-level view of payment performance: overall success rate, success/failure count, and monthly movement over time.

In [4]:
total_payments = len(df)
successful_payments = int(df["is_success"].sum())
failed_payments = int(df["is_failed"].sum())
overall_success_rate = df["is_success"].mean() * 100
overall_failure_rate = df["is_failed"].mean() * 100
failed_amount = df.loc[df["payment_status"] == "failed", "payment_amount"].sum()
estimated_recovery_cost = failed_amount * 3

html = f"""
<div style="display:flex; gap:18px; flex-wrap:wrap; font-family:Arial, sans-serif;">
  <div style="flex:1; min-width:220px; background:linear-gradient(135deg,#0f766e,#14b8a6); color:white; padding:22px; border-radius:18px; box-shadow:0 8px 22px rgba(0,0,0,.12);">
    <div style="font-size:14px; opacity:.9;">Overall Success Rate</div>
    <div style="font-size:44px; font-weight:800; margin-top:4px;">{overall_success_rate:.2f}%</div>
    <div style="font-size:13px; opacity:.9; margin-top:6px;">{successful_payments:,} successful payments</div>
  </div>
  <div style="flex:1; min-width:220px; background:#111827; color:white; padding:22px; border-radius:18px; box-shadow:0 8px 22px rgba(0,0,0,.12);">
    <div style="font-size:14px; opacity:.9;">Failure Rate</div>
    <div style="font-size:44px; font-weight:800; margin-top:4px;">{overall_failure_rate:.2f}%</div>
    <div style="font-size:13px; opacity:.9; margin-top:6px;">{failed_payments:,} failed payments</div>
  </div>
  <div style="flex:1; min-width:220px; background:#f8fafc; color:#111827; padding:22px; border-radius:18px; border:1px solid #e5e7eb; box-shadow:0 8px 22px rgba(0,0,0,.08);">
    <div style="font-size:14px; color:#64748b;">Estimated Recovery Cost Exposure</div>
    <div style="font-size:36px; font-weight:800; margin-top:4px;">£{estimated_recovery_cost:,.0f}</div>
    <div style="font-size:13px; color:#64748b; margin-top:6px;">Assumption: failures cost 3× original amount to recover</div>
  </div>
</div>
"""
display(HTML(html))

In [5]:
status_counts = (
    df["payment_status"]
    .value_counts()
    .rename_axis("payment_status")
    .reset_index(name="payment_count")
)

fig = px.bar(
    status_counts,
    x="payment_status",
    y="payment_count",
    text="payment_count",
    color="payment_status",
    title="Payment Outcome Count: Success vs Failed",
    labels={"payment_status": "Payment Status", "payment_count": "Number of Payments"},
    template="plotly_white",
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_layout(showlegend=False, yaxis_title="Payments", xaxis_title="")
fig.show()

In [6]:
fig = px.line(
    monthly_success,
    x="year_month_date",
    y="success_rate_pct",
    markers=True,
    title="Monthly Payment Success Rate Trend",
    labels={"year_month_date": "Month", "success_rate_pct": "Success Rate (%)"},
    template="plotly_white",
)
fig.update_traces(line_width=3)
fig.add_hline(
    y=overall_success_rate,
    line_dash="dash",
    annotation_text=f"Overall average: {overall_success_rate:.2f}%",
    annotation_position="bottom right",
)
fig.add_annotation(
    x=monthly_success["year_month_date"].iloc[max(0, len(monthly_success)//2)],
    y=monthly_success["success_rate_pct"].min(),
    text="Payment failures cost businesses an average of 3x<br>the original payment amount in recovery costs",
    showarrow=True,
    arrowhead=2,
    bgcolor="rgba(255,255,255,0.92)",
    bordercolor="#111827",
    borderwidth=1,
)
fig.update_layout(yaxis_ticksuffix="%", hovermode="x unified")
fig.show()

---

## 3. Temporal Patterns

This section explores whether payment timing affects success rates. Because this dataset does not include hour-level timestamps, the notebook uses `payment_day_of_month` as the timing proxy.

In [7]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
week_order = ["Week 1", "Week 2", "Week 3", "Week 4"]

temporal = (
    df.groupby(["day_of_week", "week_of_month"], observed=True)
    .agg(success_rate=("is_success", "mean"), payment_count=("payment_id", "count"))
    .reset_index()
)
temporal["success_rate_pct"] = temporal["success_rate"] * 100

heatmap_data = (
    temporal.pivot(index="day_of_week", columns="week_of_month", values="success_rate_pct")
    .reindex(day_order)
    .reindex(columns=week_order)
)

fig = px.imshow(
    heatmap_data,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale="RdYlGn",
    title="Success Rate Heatmap: Day of Week × Week of Month",
    labels=dict(x="Week of Month", y="Day of Week", color="Success Rate %"),
)
fig.update_layout(template="plotly_white")
fig.show()

In [8]:
day_of_month_success = (
    df.groupby("payment_day_of_month")
    .agg(success_rate=("is_success", "mean"), payment_count=("payment_id", "count"))
    .reset_index()
)
day_of_month_success["success_rate_pct"] = day_of_month_success["success_rate"] * 100

fig = px.line(
    day_of_month_success,
    x="payment_day_of_month",
    y="success_rate_pct",
    markers=True,
    title="Success Rate by Day of Month",
    labels={"payment_day_of_month": "Payment Day of Month", "success_rate_pct": "Success Rate (%)"},
    template="plotly_white",
)
fig.update_traces(line_width=3)
fig.update_layout(yaxis_ticksuffix="%", hovermode="x unified")
fig.show()

In [9]:
failures_by_day = (
    df.groupby("payment_day_of_month")
    .agg(failed_payments=("is_failed", "sum"), total_payments=("payment_id", "count"))
    .reset_index()
)
worst_days = failures_by_day.nlargest(5, "failed_payments")["payment_day_of_month"].tolist()
failures_by_day["risk_group"] = np.where(
    failures_by_day["payment_day_of_month"].isin(worst_days),
    "Worst 5 days",
    "Other days",
)

fig = px.bar(
    failures_by_day,
    x="payment_day_of_month",
    y="failed_payments",
    color="risk_group",
    title="Which Day of Month Has the Most Payment Failures?",
    labels={"payment_day_of_month": "Day of Month", "failed_payments": "Failed Payments", "risk_group": "Risk Group"},
    template="plotly_white",
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

worst_text = ", ".join([str(day) for day in worst_days])
display(HTML(f"""
<div style="background:#fff7ed; border-left:6px solid #f97316; padding:16px; border-radius:10px; font-family:Arial, sans-serif;">
  <b>Key temporal insight:</b> The worst-performing days by failure count are <b>{worst_text}</b>.
  These dates should be reviewed for collection timing, retry strategy, and customer balance patterns.
</div>
"""))

---

## 4. Merchant Analysis

This section identifies merchants with unusually high failure rates, high volumes, and risky merchant + mandate-age combinations.

In [10]:
merchant_summary = (
    df.groupby("merchant_id")
    .agg(
        payment_count=("payment_id", "count"),
        failed_payments=("is_failed", "sum"),
        successful_payments=("is_success", "sum"),
        avg_payment_amount=("payment_amount", "mean"),
    )
    .reset_index()
)
merchant_summary["failure_rate"] = merchant_summary["failed_payments"] / merchant_summary["payment_count"]
merchant_summary["failure_rate_pct"] = merchant_summary["failure_rate"] * 100
merchant_summary["success_rate_pct"] = 100 - merchant_summary["failure_rate_pct"]

top_failure_merchants = merchant_summary.sort_values("failure_rate_pct", ascending=False).head(10)

fig = px.bar(
    top_failure_merchants.sort_values("failure_rate_pct"),
    x="failure_rate_pct",
    y="merchant_id",
    orientation="h",
    text="failure_rate_pct",
    title="Top 10 Merchants by Failure Rate",
    labels={"failure_rate_pct": "Failure Rate (%)", "merchant_id": "Merchant"},
    template="plotly_white",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(xaxis_ticksuffix="%")
fig.show()

In [11]:
top_volume_merchants = merchant_summary.sort_values("payment_count", ascending=False).head(10)

fig = px.bar(
    top_volume_merchants.sort_values("payment_count"),
    x="payment_count",
    y="merchant_id",
    orientation="h",
    text="payment_count",
    title="Top 10 Merchants by Payment Volume",
    labels={"payment_count": "Payment Count", "merchant_id": "Merchant"},
    template="plotly_white",
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.show()

In [12]:
failure_threshold = merchant_summary["failure_rate_pct"].quantile(0.85)
volume_threshold = merchant_summary["payment_count"].quantile(0.85)
merchant_summary["outlier_flag"] = np.where(
    (merchant_summary["failure_rate_pct"] >= failure_threshold) & (merchant_summary["payment_count"] >= volume_threshold),
    "High volume + high failure",
    "Normal range",
)

fig = px.scatter(
    merchant_summary,
    x="payment_count",
    y="failure_rate_pct",
    size="failed_payments",
    color="outlier_flag",
    hover_data=["merchant_id", "successful_payments", "failed_payments", "avg_payment_amount"],
    title="Merchant Outlier Detection: Payment Volume vs Failure Rate",
    labels={"payment_count": "Payment Volume", "failure_rate_pct": "Failure Rate (%)", "outlier_flag": "Outlier Type"},
    template="plotly_white",
)
fig.add_hline(y=failure_threshold, line_dash="dash", annotation_text="85th percentile failure rate")
fig.add_vline(x=volume_threshold, line_dash="dash", annotation_text="85th percentile volume")
fig.update_layout(yaxis_ticksuffix="%")
fig.show()

In [13]:
merchant_cohort_sql = (SQL_DIR / "02_merchant_cohort_analysis.sql").read_text(encoding="utf-8")
merchant_cohort = pd.read_sql_query(merchant_cohort_sql, conn)
merchant_cohort["failure_rate"] = 100 - merchant_cohort["success_rate"]

# Focus the heatmap on the riskiest merchants by average failure rate.
risky_merchants = (
    merchant_cohort.groupby("merchant_id")
    .agg(avg_failure_rate=("failure_rate", "mean"), total_payments=("payment_count", "sum"))
    .sort_values(["avg_failure_rate", "total_payments"], ascending=[False, False])
    .head(15)
    .index
)
cohort_focus = merchant_cohort[merchant_cohort["merchant_id"].isin(risky_merchants)].copy()
cohort_pivot = cohort_focus.pivot(index="merchant_id", columns="mandate_age_band", values="success_rate")
cohort_pivot = cohort_pivot[["0-30 days", "31-90 days", "91-365 days", "365+ days"]]

fig = px.imshow(
    cohort_pivot,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale="RdYlGn",
    title="Merchant Cohort Analysis: Success Rate by Mandate Age Band",
    labels=dict(x="Mandate Age Band", y="Merchant", color="Success Rate %"),
)
fig.update_layout(template="plotly_white")
fig.show()

display(merchant_cohort.head(10))

,merchant_id,mandate_age_band,payment_count,success_rate,failure_rate
0,M172,0-30 days,14,50.00,50.00
1,M004,0-30 days,12,50.00,50.00
2,M175,0-30 days,17,52.94,47.06
3,M101,0-30 days,28,53.57,46.43
4,M052,0-30 days,18,55.56,44.44
5,M148,0-30 days,18,55.56,44.44
6,M143,0-30 days,26,57.69,42.31
7,M006,0-30 days,19,57.89,42.11
8,M005,0-30 days,24,58.33,41.67
9,M093,0-30 days,17,58.82,41.18


---

## 5. Mandate Age Impact

This section tests whether newer mandates have a higher probability of failure. This is one of the most important business questions because early mandate failures can indicate onboarding friction, customer confusion, or bank/account verification problems.

In [14]:
age_bins = [0, 30, 90, 180, 365, 730, 1095, 1460, 1825]
age_labels = ["0-30", "31-90", "91-180", "181-365", "366-730", "731-1095", "1096-1460", "1461-1825"]

df["mandate_age_bin"] = pd.cut(
    df["mandate_age_days"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True,
    right=True,
)

age_impact = (
    df.groupby("mandate_age_bin", observed=True)
    .agg(
        payment_count=("payment_id", "count"),
        failure_rate=("is_failed", "mean"),
        success_rate=("is_success", "mean"),
    )
    .reset_index()
)
age_impact["failure_rate_pct"] = age_impact["failure_rate"] * 100
age_impact["success_rate_pct"] = age_impact["success_rate"] * 100

fig = px.line(
    age_impact,
    x="mandate_age_bin",
    y="failure_rate_pct",
    markers=True,
    text="failure_rate_pct",
    title="Failure Rate by Mandate Age Band",
    labels={"mandate_age_bin": "Mandate Age Days", "failure_rate_pct": "Failure Rate (%)"},
    template="plotly_white",
)
fig.update_traces(line_width=4, texttemplate="%{text:.1f}%", textposition="top center")
fig.update_layout(yaxis_ticksuffix="%", hovermode="x unified")
fig.show()

In [15]:
new_mandate_failure = df.loc[df["mandate_age_days"] < 30, "is_failed"].mean() * 100
mature_mandate_failure = df.loc[df["mandate_age_days"] >= 365, "is_failed"].mean() * 100
failure_multiplier = new_mandate_failure / mature_mandate_failure if mature_mandate_failure > 0 else np.nan

insight_text = f"""
<div style="background:linear-gradient(135deg,#fee2e2,#fff7ed); border:1px solid #fecaca; padding:20px; border-radius:16px; font-family:Arial, sans-serif;">
  <div style="font-size:18px; font-weight:800; color:#7f1d1d;">Key Business Insight — New Mandate Risk</div>
  <div style="font-size:15px; margin-top:8px; color:#431407;">
    Payments from new mandates (&lt;30 days) fail at <b>{new_mandate_failure:.2f}%</b>, compared with <b>{mature_mandate_failure:.2f}%</b> for mature mandates (365+ days).
    That means new mandates fail about <b>{failure_multiplier:.1f}× more often</b> in this generated dataset.
  </div>
  <div style="font-size:13px; margin-top:10px; color:#7c2d12;">
    Operational action: add stronger mandate confirmation, early customer reminders, and proactive first-payment monitoring.
  </div>
</div>
"""
display(HTML(insight_text))

---

## 6. Customer Risk Segmentation

This section uses the SQL customer-risk query to identify customers with repeated payment failures and classify them into LOW, MEDIUM, HIGH, and CRITICAL risk categories.

In [16]:
high_risk_sql = (SQL_DIR / "03_high_risk_customers.sql").read_text(encoding="utf-8")
high_risk_customers = pd.read_sql_query(high_risk_sql, conn)

risk_distribution = (
    high_risk_customers["risk_category"]
    .value_counts()
    .rename_axis("risk_category")
    .reset_index(name="customer_count")
)

risk_order = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
risk_distribution["risk_category"] = pd.Categorical(risk_distribution["risk_category"], categories=risk_order, ordered=True)
risk_distribution = risk_distribution.sort_values("risk_category")

fig = px.pie(
    risk_distribution,
    names="risk_category",
    values="customer_count",
    hole=0.45,
    title="Customer Risk Category Distribution",
    template="plotly_white",
)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig.show()

In [17]:
top_20_customers = high_risk_customers.head(20).copy()
top_20_customers["failure_rate"] = (top_20_customers["failure_rate"] * 100).round(2).astype(str) + "%"
top_20_customers["total_amount_failed"] = top_20_customers["total_amount_failed"].map(lambda x: f"£{x:,.2f}")

fig = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=[f"<b>{col}</b>" for col in top_20_customers.columns],
                fill_color="#111827",
                font=dict(color="white", size=12),
                align="left",
            ),
            cells=dict(
                values=[top_20_customers[col] for col in top_20_customers.columns],
                fill_color="#f8fafc",
                align="left",
                height=26,
            ),
        )
    ]
)
fig.update_layout(title="Top 20 Highest-Risk Customers", template="plotly_white", height=720)
fig.show()

---

## 7. Geographic Analysis

This section compares success rates by bank country and bank type. It helps identify where payment failures may be linked to local banking behaviour, payment rails, or customer/account profile differences.

In [18]:
country_summary = (
    df.groupby("bank_country")
    .agg(total_payments=("payment_id", "count"), success_rate=("is_success", "mean"), avg_payment_amount=("payment_amount", "mean"))
    .reset_index()
)
country_summary["success_rate_pct"] = country_summary["success_rate"] * 100

fig = px.bar(
    country_summary.sort_values("success_rate_pct"),
    x="bank_country",
    y="success_rate_pct",
    text="success_rate_pct",
    title="Payment Success Rate by Bank Country",
    labels={"bank_country": "Bank Country", "success_rate_pct": "Success Rate (%)"},
    template="plotly_white",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(yaxis_ticksuffix="%")
fig.show()

In [19]:
bank_type_country = (
    df.groupby(["bank_country", "bank_type"])
    .agg(total_payments=("payment_id", "count"), success_rate=("is_success", "mean"), avg_payment_amount=("payment_amount", "mean"))
    .reset_index()
)
bank_type_country["success_rate_pct"] = bank_type_country["success_rate"] * 100

fig = px.bar(
    bank_type_country,
    x="bank_country",
    y="success_rate_pct",
    color="bank_type",
    barmode="group",
    text="success_rate_pct",
    title="Success Rate by Bank Type Within Each Country",
    labels={"bank_country": "Bank Country", "success_rate_pct": "Success Rate (%)", "bank_type": "Bank Type"},
    template="plotly_white",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(yaxis_ticksuffix="%")
fig.show()

In [20]:
bank_country_sql = (SQL_DIR / "04_bank_country_analysis.sql").read_text(encoding="utf-8")
bank_country_analysis = pd.read_sql_query(bank_country_sql, conn)

display(Markdown("### SQLite Banking Segment Analysis"))
display(bank_country_analysis)

### SQLite Banking Segment Analysis

,bank_country,bank_type,total_payments,success_rate,avg_payment_amount,most_common_failure_day
0,AU,credit_union,350,82.00,129.41,Tuesday
1,US,challenger,862,82.13,101.68,Monday
2,AU,challenger,1010,82.87,112.10,Monday
3,GB,challenger,6040,84.02,113.57,Monday
4,FR,challenger,1887,84.31,106.46,Monday
5,DE,high_street,4211,84.59,112.89,Monday
6,US,high_street,2362,84.63,111.19,Monday
7,DE,challenger,1565,84.92,116.25,Monday
8,DE,credit_union,529,85.07,113.76,Monday
9,US,credit_union,283,85.16,107.19,Monday


---

## 8. Business Insights Summary

- **Payment reliability is strong but still leaves meaningful recovery risk.** The generated dataset contains **50,000 payments** with an overall success rate of **85.31%** and **7,346 failed payments**.
- **Failed payments create a hidden cost problem.** Using the 3× recovery-cost assumption, failed payment value creates an estimated recovery exposure of around **£3,025,192**.
- **Timing matters.** **Monday** has the highest observed failure rate at **18.99%**, and day **3** has the highest failure count by day of month.
- **Merchant performance is not equal.** Merchant **M068** has the highest failure rate at **23.63%**, so merchant-level monitoring should be part of payment operations.
- **New mandates are riskier than mature mandates.** Mandates under 30 days fail at **23.58%**, around **1.8×** the failure rate of 365+ day mandates, making onboarding and first-payment monitoring a priority.
- **Customer risk is concentrated.** The SQL risk model flags **37 CRITICAL** and **325 HIGH** risk customers, giving teams a practical shortlist for proactive outreach, retry strategy, or account review.
- **Banking geography also matters.** **US** has the lowest country-level success rate at **84.06%**, so country and bank-type segmentation can help target payment recovery improvements.

In [21]:
# Close SQLite connection when analysis is complete.
conn.close()
print("EDA notebook complete. SQLite connection closed.")

EDA notebook complete. SQLite connection closed.
